# FER-2013 Data Exploration
Quick look at the dataset: class distribution, sample images, and augmentation preview.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
from src.data.dataset import FER2013Dataset
from src.data.transforms import get_train_transforms, get_eval_transforms

DATA_ROOT = '../data/fer2013'

In [ ]:
# Load dataset without transforms to inspect raw PIL images
train_ds = FER2013Dataset(DATA_ROOT, split='train')
test_ds  = FER2013Dataset(DATA_ROOT, split='test')

print('Classes:', train_ds.classes)
print('Train samples:', len(train_ds))
print('Test samples: ', len(test_ds))

In [ ]:
# Class distribution
counts = train_ds.get_class_counts()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(train_ds.classes, counts)
ax.bar_label(bars)
ax.set_title('Training set class distribution')
ax.set_ylabel('Number of images')
plt.tight_layout()
plt.show()

print('\nClass frequencies:')
total = sum(counts)
for cls, cnt in zip(train_ds.classes, counts):
    print(f'  {cls:<10} {cnt:>5}  ({cnt/total*100:.1f}%)')

In [ ]:
# Sample images — one per class
from collections import defaultdict

class_samples = defaultdict(list)
for img, label in train_ds:
    if len(class_samples[label]) < 4:
        class_samples[label].append(img)
    if all(len(v) >= 4 for v in class_samples.values()):
        break

fig, axes = plt.subplots(len(train_ds.classes), 4, figsize=(8, 2 * len(train_ds.classes)))
for row, cls in enumerate(train_ds.classes):
    idx = train_ds.class_to_idx[cls]
    for col, img in enumerate(class_samples[idx]):
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls, fontsize=9)
plt.suptitle('Sample images per class')
plt.tight_layout()
plt.show()

In [ ]:
# Augmentation preview
import torchvision.transforms.functional as TF

raw_img, label = train_ds[0]
train_tf = get_train_transforms(image_size=48, channels=1)

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
axes[0].imshow(raw_img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
for i in range(1, 6):
    aug = train_tf(raw_img)
    axes[i].imshow(aug.squeeze(), cmap='gray')
    axes[i].set_title(f'Aug {i}')
    axes[i].axis('off')
plt.suptitle(f'Augmentation samples — class: {train_ds.classes[label]}')
plt.tight_layout()
plt.show()